# Batch Paint-by-Numbers Generator

Use this for 10, 50, or 100 images at once.

What it does:
- Upload many images or one zip.
- Generate numbered outline PNGs.
- Save colored reference PNGs.
- Save palette JSON files.
- Download everything as one zip.

For best results with generated colored art, use clean flat-color images with clear separated shapes. For existing black-and-white line art, turn on `LINE_ART_MODE` in the settings cell.

In [ ]:
!pip -q install pillow numpy


In [ ]:
%%writefile pbn_batch_pipeline.py
from pathlib import Path
from collections import deque
from dataclasses import dataclass
import json, math, re, shutil, zipfile
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont, ImageOps

IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.webp'}

@dataclass
class Region:
    number: int
    label: int
    color: tuple
    pixels: list
    center: tuple
    bbox: tuple
    @property
    def area(self): return len(self.pixels)

def slugify(value):
    return re.sub(r'[^A-Za-z0-9]+', '_', value).strip('_').lower() or 'paint_by_numbers'

def load_image(path, max_size):
    image = Image.open(path).convert('RGB')
    if max_size > 0:
        image.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
    return image

def quantize_image(image, colors, smooth_radius):
    work = image.filter(ImageFilter.GaussianBlur(radius=smooth_radius)) if smooth_radius > 0 else image
    return work.quantize(colors=colors, method=Image.Quantize.MEDIANCUT).convert('RGB')

def build_label_map(reference):
    pixels = np.asarray(reference, dtype=np.uint8)
    h, w, _ = pixels.shape
    labels = np.zeros((h, w), dtype=np.int32)
    palette, color_to_label = [], {}
    for y in range(h):
        for x in range(w):
            color = tuple(int(v) for v in pixels[y, x])
            label = color_to_label.get(color)
            if label is None:
                label = len(palette)
                color_to_label[color] = label
                palette.append(color)
            labels[y, x] = label
    return labels, palette

def region_label_point(pixels, bbox):
    mean_x = sum(p[0] for p in pixels) / len(pixels)
    mean_y = sum(p[1] for p in pixels) / len(pixels)
    min_x, min_y, max_x, max_y = bbox
    pixel_set = set(pixels)
    q, dist = deque(), {}
    for x, y in pixels:
        boundary = x in (min_x, max_x) or y in (min_y, max_y)
        if not boundary:
            boundary = any((nx, ny) not in pixel_set for nx, ny in ((x+1,y),(x-1,y),(x,y+1),(x,y-1)))
        if boundary:
            dist[(x, y)] = 0
            q.append((x, y))
    while q:
        x, y = q.popleft()
        nd = dist[(x, y)] + 1
        for nx, ny in ((x+1,y),(x-1,y),(x,y+1),(x,y-1)):
            p = (nx, ny)
            if p in pixel_set and p not in dist:
                dist[p] = nd
                q.append(p)
    return min(pixels, key=lambda p: (-dist.get(p, 0), (p[0]-mean_x)**2 + (p[1]-mean_y)**2))

def connected_regions(labels, palette, min_area, include_background, exclude_dark_regions):
    h, w = labels.shape
    visited = np.zeros((h, w), dtype=bool)
    regions, n = [], 1
    for y in range(h):
        for x in range(w):
            if visited[y, x]: continue
            label = int(labels[y, x])
            q = deque([(x, y)])
            visited[y, x] = True
            pixels = []
            min_x = max_x = x; min_y = max_y = y
            while q:
                cx, cy = q.popleft()
                pixels.append((cx, cy))
                min_x = min(min_x, cx); max_x = max(max_x, cx)
                min_y = min(min_y, cy); max_y = max(max_y, cy)
                for nx, ny in ((cx+1,cy),(cx-1,cy),(cx,cy+1),(cx,cy-1)):
                    if nx < 0 or ny < 0 or nx >= w or ny >= h: continue
                    if visited[ny, nx] or int(labels[ny, nx]) != label: continue
                    visited[ny, nx] = True
                    q.append((nx, ny))
            color = palette[label]
            touches_border = min_x == 0 or min_y == 0 or max_x == w - 1 or max_y == h - 1
            near_white = all(c >= 235 for c in color)
            near_black = all(c <= 45 for c in color)
            if len(pixels) < min_area: continue
            if touches_border and near_white and not include_background: continue
            if exclude_dark_regions and near_black: continue
            bbox = (min_x, min_y, max_x, max_y)
            regions.append(Region(n, label, color, pixels, region_label_point(pixels, bbox), bbox))
            n += 1
    return regions

def make_boundary_mask(labels, outline_width):
    h, w = labels.shape
    edges = np.zeros((h, w), dtype=np.uint8)
    edges[:,1:] |= labels[:,1:] != labels[:,:-1]
    edges[:,:-1] |= labels[:,1:] != labels[:,:-1]
    edges[1:,:] |= labels[1:,:] != labels[:-1,:]
    edges[:-1,:] |= labels[1:,:] != labels[:-1,:]
    mask = Image.fromarray(edges * 255, mode='L')
    if outline_width > 1:
        mask = mask.filter(ImageFilter.MaxFilter(outline_width | 1))
    return mask

def dark_source_mask(source, line_width):
    mask = ImageOps.grayscale(source).point(lambda v: 255 if v < 80 else 0, mode='L')
    if line_width > 1:
        mask = mask.filter(ImageFilter.MaxFilter(line_width | 1))
    return mask

def font_for(region, scale):
    min_x, min_y, max_x, max_y = region.bbox
    box = min(max_x - min_x + 1, max_y - min_y + 1)
    size = max(10, min(38, int(math.sqrt(region.area) * 0.28), box // 2)) * scale
    for name in ('DejaVuSans-Bold.ttf', 'arial.ttf'):
        try: return ImageFont.truetype(name, size=size)
        except OSError: pass
    return ImageFont.load_default()

def draw_outline(labels, regions, output_path, outline_width, number_min_area, source=None, line_art_mode=False, antialias_scale=3):
    h, w = labels.shape
    base_mask = make_boundary_mask(labels, outline_width)
    if line_art_mode and source is not None:
        base_mask = Image.fromarray(np.maximum(np.asarray(base_mask), np.asarray(dark_source_mask(source, outline_width))).astype(np.uint8), mode='L')
    scale = max(1, int(antialias_scale))
    canvas = Image.new('RGB', (w * scale, h * scale), 'white')
    mask = base_mask.resize((w * scale, h * scale), Image.Resampling.LANCZOS)
    canvas.paste('black', mask=mask)
    draw = ImageDraw.Draw(canvas)
    for region in regions:
        if region.area < number_min_area: continue
        text = str(region.number)
        font = font_for(region, scale)
        bbox = draw.textbbox((0, 0), text, font=font)
        x = region.center[0] * scale - (bbox[2] - bbox[0]) // 2
        y = region.center[1] * scale - (bbox[3] - bbox[1]) // 2
        draw.text((x, y), text, fill='black', font=font)
    canvas.save(output_path)

def save_metadata(path, source, reference, outline, regions, size, colors, min_area):
    data = {
        'source': str(source), 'outline_png': outline.name, 'colored_reference_png': reference.name,
        'image_width': size[0], 'image_height': size[1], 'max_colors': colors,
        'min_region_area': min_area, 'region_count': len(regions),
        'regions': [
            {'number': r.number, 'hex': '#{:02X}{:02X}{:02X}'.format(*r.color), 'rgb': list(r.color),
             'center': {'x': r.center[0], 'y': r.center[1]}, 'area': r.area,
             'bbox': {'x_min': r.bbox[0], 'y_min': r.bbox[1], 'x_max': r.bbox[2], 'y_max': r.bbox[3]}}
            for r in regions]
    }
    path.write_text(json.dumps(data, indent=2), encoding='utf-8')

def process_one(input_path, output_dir, colors=12, min_region_area=160, outline_width=1, number_min_area=420, max_size=1024, smooth_radius=0.8, line_art_mode=False, include_background=False, antialias_scale=3, save_original_reference=True):
    input_path = Path(input_path)
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    name = slugify(input_path.stem)
    source = load_image(input_path, max_size)
    quantized = quantize_image(source, colors, smooth_radius)
    labels, palette = build_label_map(quantized)
    regions = connected_regions(labels, palette, min_region_area, include_background, line_art_mode)
    ref_path = output_dir / f'{name}_colored_reference.png'
    out_path = output_dir / f'{name}_numbered_outline.png'
    json_path = output_dir / f'{name}_palette.json'
    (source if save_original_reference else quantized).save(ref_path)
    draw_outline(labels, regions, out_path, outline_width, number_min_area, source=source, line_art_mode=line_art_mode, antialias_scale=antialias_scale)
    save_metadata(json_path, input_path, ref_path, out_path, regions, source.size, colors, min_region_area)
    return {'input': input_path.name, 'regions': len(regions), 'outline': out_path, 'reference': ref_path, 'json': json_path}

def collect_images(upload_dir):
    upload_dir = Path(upload_dir)
    for zip_path in upload_dir.glob('*.zip'):
        target = upload_dir / zip_path.stem
        target.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(target)
    return sorted(p for p in upload_dir.rglob('*') if p.suffix.lower() in IMAGE_EXTS)


In [ ]:
from google.colab import files
from pathlib import Path
import shutil

UPLOAD_DIR = Path('input_images')
UPLOAD_DIR.mkdir(exist_ok=True)

print('Upload multiple images, or upload one zip containing your images.')
uploaded = files.upload()
for name, data in uploaded.items():
    (UPLOAD_DIR / name).write_bytes(data)

print('Uploaded files:', len(uploaded))


In [ ]:
from pbn_batch_pipeline import collect_images, process_one
from google.colab import files
from pathlib import Path
import shutil, json

# SETTINGS
# For 100 generated colored images:
LINE_ART_MODE = False
COLORS = 12

# For existing black-and-white mandala line art, use:
# LINE_ART_MODE = True
# COLORS = 2

OUTPUT_DIR = Path('generated_image')
MAX_SIZE = 1536
OUTLINE_WIDTH = 1
ANTIALIAS_SCALE = 3
SMOOTH_RADIUS = 0.8
MIN_REGION_AREA = 220
NUMBER_MIN_AREA = 700
SAVE_ORIGINAL_REFERENCE = True

images = collect_images('input_images')
print('Images found:', len(images))

results = []
for i, image_path in enumerate(images, 1):
    print(f'[{i}/{len(images)}] Processing {image_path.name}...')
    result = process_one(
        image_path,
        OUTPUT_DIR,
        colors=COLORS,
        min_region_area=MIN_REGION_AREA,
        outline_width=OUTLINE_WIDTH,
        number_min_area=NUMBER_MIN_AREA,
        max_size=MAX_SIZE,
        smooth_radius=SMOOTH_RADIUS,
        line_art_mode=LINE_ART_MODE,
        antialias_scale=ANTIALIAS_SCALE,
        save_original_reference=SAVE_ORIGINAL_REFERENCE,
    )
    results.append({'input': result['input'], 'regions': result['regions']})

(OUTPUT_DIR / '_batch_summary.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
zip_path = shutil.make_archive('generated_image', 'zip', OUTPUT_DIR)
print('Done. Zip ready:', zip_path)
files.download(zip_path)


## Tuning

If outlines are too thick: lower `OUTLINE_WIDTH` to `1`.

If output is jagged: keep `ANTIALIAS_SCALE = 3` or try `4`, and keep `MAX_SIZE` at `1536` or higher.

If there are too many tiny regions: increase `MIN_REGION_AREA`, or reduce `COLORS`.

If there are too few regions: decrease `MIN_REGION_AREA`, or increase `COLORS`.

For colored generated images, keep `SAVE_ORIGINAL_REFERENCE = True`. This preserves the clean original as the reference image instead of saving the quantized processing image.